In [1]:
%pip install -q pandas


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import importlib

import pandas as pd
import src.constants

importlib.reload(src.constants)
from src.constants import (
    Column,
    MODEL_FEATURE_COLUMNS,
    OUTPUT_DIR,
)

In [3]:
def calculate_ratios(df: pd.DataFrame):     
    df[Column.ADJECTIVE_RATIO] = df[Column.ADJECTIVE_COUNT] / df[Column.WORD_COUNT]
    df[Column.VERB_RATIO] = df[Column.VERB_COUNT] / df[Column.WORD_COUNT]
    df[Column.SUPERLATIVE_RATIO] = df[Column.SUPERLATIVE_COUNT] / df[Column.WORD_COUNT]

    df[Column.FIRST_PERSON_PRONOUN_RATIO] = (
        df[Column.FIRST_PERSON_PRONOUN_COUNT] / df[Column.WORD_COUNT]
    )
    df[Column.THIRD_PERSON_PRONOUN_RATIO] = (
        df[Column.THIRD_PERSON_PRONOUN_COUNT] / df[Column.WORD_COUNT]
    )

    claim_count = (
        df[Column.SUBJECTIVE_CLAIM_COUNT]
        + df[Column.OBJECTIVE_CLAIM_COUNT]
    )
    df[Column.SUBJECTIVITY] = (
        df[Column.SUBJECTIVE_CLAIM_COUNT] / claim_count.mask(claim_count == 0)
    ).fillna(0.5)

    normalized_claim_count = claim_count.mask(claim_count == 0)
    df[Column.EXPERIENTIAL_DETAIL] = (
        df[Column.EXPERIENTIAL_DETAIL_CLAIM_COUNT] / normalized_claim_count
    ).fillna(0).clip(0, 1)
    df[Column.POSITIVE_AFFECT] = (
        df[Column.POSITIVE_AFFECT_CLAIM_COUNT] / normalized_claim_count
    ).fillna(0).clip(0, 1)
    df[Column.NEGATIVE_AFFECT] = (
        df[Column.NEGATIVE_AFFECT_CLAIM_COUNT] / normalized_claim_count
    ).fillna(0).clip(0, 1)
    df[Column.UNCERTAINTY] = (
        df[Column.UNCERTAIN_CLAIM_COUNT] / normalized_claim_count
    ).fillna(0).clip(0, 1)
    df[Column.CATEGORY_SPECIFICITY] = (
        df[Column.CATEGORY_SPECIFIC_CLAIM_COUNT] / normalized_claim_count
    ).fillna(0).clip(0, 1)

    text_sentiment_normalized = df[Column.TEXT_SENTIMENT] / 2
    rating_sentiment = (df[Column.RATING] - 3) / 2
    df[Column.INTERNAL_CONSISTENCY] = (
        1 - (text_sentiment_normalized - rating_sentiment).abs() / 2
    ).clip(0, 1)

    print(df.columns)
    print(df.head())

def merge(
    basic_features_dir,
    output_llm_dir,
    output_dir,
    include_label=False,
    include_split=False,
):
    deterministics_df = pd.read_csv(basic_features_dir)
    output_llm = pd.read_csv(output_llm_dir)

    final = pd.merge(
        deterministics_df,
        output_llm,
        how="inner",
        on=Column.ID,
        validate="one_to_one",
    )
    if len(final) != len(deterministics_df) or len(final) != len(output_llm):
        raise ValueError(
            "Basic- und LLM-Features enthalten nicht dieselben IDs. "
            "Prüfe fehlgeschlagene LLM-Extraktionen."
        )

    calculate_ratios(final)

    final_columns = [Column.ID]
    if include_label:
        final_columns.append(Column.LABEL)
    if include_split:
        final_columns.append(Column.SPLIT)
    final_columns.extend(MODEL_FEATURE_COLUMNS)
    final = final[final_columns].copy()
    final.to_csv(output_dir, index=None)

In [4]:
basic_features_dir = OUTPUT_DIR / "train" / "basic_features.csv"
output_llm_dir = OUTPUT_DIR / "train" / "llm_features.csv"
output_dir = OUTPUT_DIR / "train" / "final.csv"

basic_features_dir2 = OUTPUT_DIR / "sample" / "basic_features.csv"
output_llm_dir2 = OUTPUT_DIR / "sample" / "llm_features.csv"
output_dir2 = OUTPUT_DIR / "sample" / "final.csv"

merge(
    basic_features_dir,
    output_llm_dir,
    output_dir,
    include_label=True,
    include_split=True,
)
merge(
    basic_features_dir2,
    output_llm_dir2,
    output_dir2,
    include_label=False,
    include_split=False,
)

ValueError: Basic- und LLM-Features enthalten nicht dieselben IDs. Prüfe fehlgeschlagene LLM-Extraktionen.